In [1]:
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm
import json

OUTPUT_DIR = Path("sentence_pairs_data")
OUTPUT_OPUS100 = OUTPUT_DIR / "cleaned_pairs_en_id_opus100.jsonl"
OUTPUT_NLLB = OUTPUT_DIR / "cleaned_pairs_en_id_nllb_20k.jsonl"

def is_valid_sentence(text, min_chars=50, max_chars=200000):
    if text is None:
        return False
    text = text.strip()
    if len(text) < min_chars:
        return False
    if len(text) > max_chars:
        return False
    if text.count(" ") == 0 and len(text) > 30:
        return False
    return True
 
def get_opus100(max_samples=None):
    OUTPUT_OPUS100.parent.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset("Helsinki-NLP/opus-100", "en-id", streaming=True)

    written = 0
    seen = 0
    with OUTPUT_OPUS100.open("w", encoding="utf-8") as f:
        for split in dataset:
            print(f"Processing split: {split}")

            for row in tqdm(dataset[split]):
                pair = row["translation"]

                en = pair.get("en")
                id_ = pair.get("id")

                if not is_valid_sentence(en) or not is_valid_sentence(id_):
                    continue
                
                item = {
                    "src_lang": "en",
                    "tgt_lang": "id",
                    "source":en,
                    "target":id_,
                    "source_dataset":"OPUS-100",
                    "split":split,
                }

                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                written += 1

                if max_samples is not None and written >= max_samples:
                    print(f"Saved {written} rows to {OUTPUT_OPUS100}")
                    return
                
    print(f"Saved {written} rows to {OUTPUT_OPUS100}")

def get_nllb(max_samples=None):
    OUTPUT_NLLB.parent.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset("allenai/nllb", "eng_Latn-ind_Latn", streaming=True, trust_remote_code=True,)

    written = 0
    seen = 0

    with OUTPUT_NLLB.open("w", encoding="utf-8") as f:
        for split in dataset:
            print(f"Processing split: {split}")
            i = 0
            for row in tqdm(dataset[split]):
                i += 1
                if i < 110000:
                    continue

                pair = row["translation"]

                en = pair.get("eng_Latn")
                id_ = pair.get("ind_Latn")

                if not is_valid_sentence(en) or not is_valid_sentence(id_):
                    continue
                
                item = {
                    "src_lang": "en",
                    "tgt_lang": "id",
                    "source":en,
                    "target":id_,
                    "source_dataset":"NLLB",
                    "split":split,
                }

                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                written += 1

                if max_samples is not None and written >= max_samples:
                    print(f"Saved {written} rows to {OUTPUT_NLLB}")
                    return
                
    print(f"Saved {written} rows to {OUTPUT_NLLB}")

# get_opus100(100000)
get_nllb(20000)


                

/Users/nicholas/miniconda3/envs/nllb-data/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


Processing split: train


133704it [00:27, 4789.89it/s] 


Saved 20000 rows to sentence_pairs_data/cleaned_pairs_en_id_nllb_20k.jsonl


In [1]:
# Load Dataset
import pandas as pd
from datasets import Dataset, DatasetDict

DATA_PATH = "sentence_pairs_data/synthetic_nllb_10000_id70_en30_to_bbc_googletrans.jsonl"
SEED = 42

df = pd.read_json(DATA_PATH, lines=True)

full_dataset = Dataset.from_pandas(df, preserve_index=False)
split_dataset = full_dataset.train_test_split(test_size=0.05, seed=SEED, shuffle=True)

dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))
print(dataset["train"][0])

import re
# Clean Text
def clean_text(ex):
    ex = str(ex)
    ex = ex.strip()
    ex = re.sub(r"\s+", " ", ex)
    return ex

def clean_example(ex):
    ex["source"] = clean_text(ex["source"])
    ex["target"] = clean_text(ex["target"])
    return ex

dataset = dataset.map(clean_example)
print("="*80)
print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))

# Filter empty example
def valid_example(ex):
    return len(ex["source"]) > 0 and len(ex["target"]) > 0

dataset = dataset.filter(valid_example)
print("="*80)
print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))

# Filter long sentence
def length_filter(ex, max_chars=1000, ratio_limit=3.0):
    src = ex["source"]
    tgt = ex["target"]

    if len(src) > max_chars or len(tgt) > max_chars:
        return False

    src_len = max(len(src.split()),1)
    tgt_len = max(len(tgt.split()),1)
    ratio = max(src_len / tgt_len, tgt_len / src_len)
    return ratio <= ratio_limit

dataset = dataset.filter(length_filter)
print("="*80)
print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))

# remove exact duplicate
def deduplicate_split(ex):
    seen = set()
    keep = []
    for x in ex:
        pair = (x["source"], x["target"])
        if pair not in seen:
            seen.add(pair)
            keep.append(x)
    return keep

dataset["train"] = deduplicate_split(dataset["train"])
dataset["validation"] = deduplicate_split(dataset["validation"])

dataset = DatasetDict({
    "train": Dataset.from_list(dataset["train"]),
    "validation": Dataset.from_list(dataset["validation"])
})

# print(dataset)
print("="*80)
print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))
print(dataset["train"])

/Users/nicholas/miniconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train examples: 9500
Validation examples: 500
{'src_lang': 'id', 'tgt_lang': 'bbc', 'source': 'Hotel ini membuat makanan Arab yang terbaik; makanan adalah pengalaman sendiri.', 'target': 'Hotel on mambahen sipanganon Arab na dumenggan; sipanganon i ma pengalaman di dirina.', 'source_dataset': 'NLLB', 'generation_method': 'googletrans', 'split': 'train', 'original_en': 'These hotels make the best Arabian food; the food is an experience of its own.', 'original_id': 'Hotel ini membuat makanan Arab yang terbaik; makanan adalah pengalaman sendiri.'}


Map: 100%|██████████| 500/500 [00:00<00:00, 22449.84 examples/s]


Train examples: 9500
Validation examples: 500


Filter: 100%|██████████| 500/500 [00:00<00:00, 152298.62 examples/s]


Train examples: 9500
Validation examples: 500


Filter: 100%|██████████| 500/500 [00:00<00:00, 58527.35 examples/s]


Train examples: 9500
Validation examples: 500
Train examples: 9465
Validation examples: 500
Dataset({
    features: ['src_lang', 'tgt_lang', 'source', 'target', 'source_dataset', 'generation_method', 'split', 'original_en', 'original_id'],
    num_rows: 9465
})


### Get sentences from MADLAD-400

In [2]:
from datasets import load_dataset
import pandas as pd
import re
from pathlib import Path

# ============================================================
# Config
# ============================================================

url = "https://huggingface.co/datasets/allenai/MADLAD-400/resolve/main/data/bbc/bbc_clean_0000.jsonl.gz"

OUTPUT_PATH = Path("sentence_pairs_data/madlad_bbc_sentences.jsonl")

MIN_CHARS = 5
MAX_CHARS = 1000

# ============================================================
# Load MADLAD Batak Toba monolingual data
# ============================================================

raw = load_dataset(
    "json",
    data_files={"train": url},
    split="train",
)

print(raw)
print(raw.column_names)
print(raw[0])

text_col = "text"

# ============================================================
# Cleaning functions
# ============================================================

def normalize_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # Convert escaped newline into real sentence boundary
    text = text.replace("\\n", "\n")

    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Put spaces around newlines so they are not glued
    text = re.sub(r"\n+", "\n", text)

    # Normalize spaces/tabs, but preserve newline for sentence splitting
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


def split_into_sentences(text: str):
    text = normalize_text(text)

    if not text:
        return []

    # First split on newlines
    chunks = re.split(r"\n+", text)

    sentences = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        # Then split on punctuation followed by whitespace
        parts = re.split(r"(?<=[.!?])\s+", chunk)

        for sent in parts:
            sent = re.sub(r"\s+", " ", sent).strip()

            if not sent:
                continue

            if len(sent) < MIN_CHARS:
                continue

            if len(sent) > MAX_CHARS:
                continue

            sentences.append(sent)

    return sentences


# ============================================================
# Convert document rows into sentence rows
# ============================================================

rows = []

for i, example in enumerate(raw):
    text = example[text_col]
    sentences = split_into_sentences(text)

    for sent_id, sentence in enumerate(sentences):
        rows.append({
            "id": f"madlad_bbc_{i}_{sent_id}",
            "lang": "bbc_Latn",
            "text": sentence,
            "source": "madlad-400",
        })

df = pd.DataFrame(rows)

print("Before dedup:", len(df))

# Deduplicate exact duplicate sentences
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("After dedup:", len(df))

# Optional: sort by length to inspect
df["length"] = df["text"].str.len()

print(df.head(20))
print(df["length"].describe())

# Remove helper column before saving
df = df.drop(columns=["length"])

# ============================================================
# Save
# ============================================================

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"Saved sentence-level data to: {OUTPUT_PATH}")

Dataset({
    features: ['text'],
    num_rows: 1274
})
['text']
{'text': 'Jamita Evangelium, 15 September 2019, Minggu XIII Dung Trinitatis "Mangolu di Bagasan Serep ni Roha" (Lukas 14: 7-11) Pelita Batak Online\\nDi tongatonga ni parsaoran hajolmaon sian narobi sahat tu nuaeng on, sai dipodahon angka natuatua dohot pangajari do hinaringkot ni serep ni roha, asa tubu parsaoran nauli sama jolma i. Laos dipodahon do tong asa pasidingon ni jolma i na maralo tu serep ni roha i, i ma ginjang ni roha, ai sega do parsaoran nauli binahen ni ginjang ni roha. Nang di tongatonga ni parsaoran ni bangso ni Debata, Israel, songon na nalnal di buku habisuhon dipodahon do asa didungdung bangso i serep ni roha, jala dipasiding ginjang ni roha. Laho mangondolhon i, tangkas ma didok na dialo Debata do ginjang ni roha. "Ibana mamarengkelengkelihon halak pangelesi, alai dilehon do asi ni roha anggo tu halak na unduk marroha"(Poda 3: 34).\\nSongon i do nang di tongatonga ni parsaoran Hakristenon, Israel na

In [1]:
from pathlib import Path
import re
import pandas as pd

from datasets import load_dataset
from sacremoses import MosesPunctNormalizer
from wtpsplit import SaT


# ============================================================
# Config
# ============================================================

URL = "https://huggingface.co/datasets/allenai/MADLAD-400/resolve/main/data/bbc/bbc_clean_0000.jsonl.gz"

OUTPUT_PATH = Path("sentence_pairs_data/madlad_bbc_sentences_wtpsplit_clean.jsonl")

LANG = "bbc_Latn"
SOURCE_NAME = "madlad-400"

MIN_CHARS = 5
MAX_CHARS = 300
MIN_WORDS = 2

# Use smaller/faster model first.
# You can also try: "sat-12l-sm" for stronger segmentation.
SAT_MODEL_NAME = "sat-3l-sm"

# Batak / Indonesian / common abbreviations that should not end a sentence
ABBREVIATIONS = {
    # Batak / church / title / name-related
    "Gr.", "Lbn.", "Pdt.", "Op.", "Br.", "St.", "Amang.", "Inang.",

    # Indonesian/common titles
    "Dr.", "Ir.", "Prof.", "H.", "Hj.", "Ny.", "Tn.", "Bpk.", "Bp.",
    "Sdr.", "Sdri.",

    # Places/admin
    "Jl.", "No.", "Ds.", "Kec.", "Kab.", "Prov.",

    # Common lowercase abbreviations
    "dll.", "dst.", "dsb.", "alm.",
}

DOT_PLACEHOLDER = "<DOT>"


# ============================================================
# Load tools
# ============================================================

mpn = MosesPunctNormalizer(lang="en")
sat = SaT(SAT_MODEL_NAME)


# ============================================================
# Text normalization
# ============================================================

def normalize_raw_text(text: str) -> str:
    """
    Normalize document-level raw text before segmentation.
    Important: preserve newlines because they can be real boundaries,
    but convert escaped \\n to real newlines first.
    """
    if text is None:
        return ""

    text = str(text)

    # Convert literal escaped newline to real newline
    text = text.replace("\\n", "\n")

    # Normalize newline variants
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Moses punctuation normalization
    # This is useful for quotes, unicode punctuation, spacing around punctuation, etc.
    text = mpn.normalize(text)

    # Normalize spaces/tabs but preserve newlines
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse too many newlines
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def normalize_sentence(text: str) -> str:
    """
    Normalize sentence-level text after segmentation/repair.
    """
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\\n", " ")
    text = text.replace("\n", " ")

    text = mpn.normalize(text)

    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


# ============================================================
# Abbreviation protection
# ============================================================

def protect_abbreviations(text: str) -> str:
    """
    Temporarily replace abbreviation periods with a placeholder
    so sentence splitting does not split after them.
    """
    for abbr in sorted(ABBREVIATIONS, key=len, reverse=True):
        protected = abbr.replace(".", DOT_PLACEHOLDER)

        text = re.sub(
            rf"(?<!\w){re.escape(abbr)}",
            protected,
            text,
        )

    return text


def restore_abbreviations(text: str) -> str:
    return text.replace(DOT_PLACEHOLDER, ".")


# ============================================================
# Custom repair rules
# ============================================================

def ends_with_known_abbreviation(text: str) -> bool:
    text = restore_abbreviations(text).strip()

    abbr_no_dot = [re.escape(a[:-1]) for a in ABBREVIATIONS if a.endswith(".")]
    pattern = r"\b(" + "|".join(abbr_no_dot) + r")\.$"

    return bool(re.search(pattern, text))


def looks_like_name_fragment(text: str) -> bool:
    """
    Detect fragments like:
    Frederik Lbn.
    Gaol sian Buntu Raja
    """
    text = restore_abbreviations(text).strip()

    if not text:
        return False

    words = text.split()

    if len(words) > 6:
        return False

    # Allow lowercase connectors commonly found in names/phrases
    allowed_lower = {"sian", "di", "ni", "tu", "bin", "boru", "br"}

    good = 0

    for w in words:
        clean = re.sub(r"[^\w.'-]", "", w)

        if not clean:
            continue

        # Capitalized name token
        if re.match(r"^[A-Z][A-Za-z.'-]*\.?$", clean):
            good += 1
            continue

        # Known lowercase connector
        if clean.lower() in allowed_lower:
            good += 1
            continue

        # Known abbreviation
        if clean + "." in ABBREVIATIONS or clean in {a.rstrip(".") for a in ABBREVIATIONS}:
            good += 1
            continue

    return good >= max(1, len(words) - 1)


def should_merge(prev: str, curr: str) -> bool:
    """
    Decide whether two candidate segments should be merged.

    Handles:
    Bendahara : Gr.
    Frederik Lbn.
    Gaol sian Buntu Raja

    Also handles cases where wtpsplit/newline splits after an abbreviation.
    """
    prev_restored = restore_abbreviations(prev).strip()
    curr_restored = restore_abbreviations(curr).strip()

    if not prev_restored or not curr_restored:
        return False

    # Previous ends with known abbreviation/title/name abbreviation
    if ends_with_known_abbreviation(prev_restored):
        return True

    # Role/title marker before name
    if re.search(r":\s*(Gr|Dr|Ir|Prof|Pdt|Op|St|Bpk|Bp)\.$", prev_restored):
        return True

    # Previous looks incomplete and current looks like a name fragment
    if prev_restored.endswith(":") and looks_like_name_fragment(curr_restored):
        return True

    # Current is a short name-like continuation after a role/title line
    if re.search(r":", prev_restored) and looks_like_name_fragment(curr_restored):
        return True

    return False


def repair_segments(segments):
    """
    Merge over-split segments produced by newline splitting or wtpsplit.
    """
    repaired = []

    for seg in segments:
        seg = seg.strip()
        if not seg:
            continue

        if not repaired:
            repaired.append(seg)
            continue

        if should_merge(repaired[-1], seg):
            repaired[-1] = repaired[-1].rstrip() + " " + seg.lstrip()
        else:
            repaired.append(seg)

    return repaired


# ============================================================
# Filtering
# ============================================================

def is_bad_sentence(sent: str) -> bool:
    if not sent:
        return True

    if len(sent) < MIN_CHARS:
        return True

    if len(sent) > MAX_CHARS:
        return True

    if len(sent.split()) < MIN_WORDS:
        return True

    # Remove URLs, emails, HTML
    if re.search(r"http\S+|www\.\S+|\S+@\S+|<[^>]+>", sent, flags=re.IGNORECASE):
        return True

    # Remove mostly punctuation/symbol garbage
    punct_count = sum(1 for c in sent if not c.isalnum() and not c.isspace())
    punct_ratio = punct_count / max(len(sent), 1)

    if punct_ratio > 0.35:
        return True

    # Remove rows with very little alphabetic content
    alpha_count = sum(1 for c in sent if c.isalpha())
    alpha_ratio = alpha_count / max(len(sent), 1)

    if alpha_ratio < 0.45:
        return True

    return False


# ============================================================
# Sentence segmentation
# ============================================================

def split_document_to_sentences(text: str):
    """
    Main sentence splitter:
    1. Normalize raw text
    2. Protect abbreviations
    3. Use wtpsplit/SaT
    4. Repair over-splits
    5. Restore abbreviations
    6. Clean/filter
    """
    text = normalize_raw_text(text)

    if not text:
        return []

    protected_text = protect_abbreviations(text)

    # SaT segmentation.
    # We keep newlines in text because document line breaks are often meaningful
    # in MADLAD/Wikipedia-like data.
    try:
        segments = sat.split(protected_text)
    except TypeError:
        # Compatibility fallback in case your wtpsplit version has slightly different args
        segments = sat.split(protected_text)

    # SaT can return nested output depending on input shape/version.
    # Flatten defensively.
    if segments and isinstance(segments[0], list):
        segments = [x for sub in segments for x in sub]

    segments = [str(x).strip() for x in segments if str(x).strip()]

    # Additional split on newlines inside segments, because some boundaries
    # may remain inside one SaT segment.
    newline_split_segments = []

    for seg in segments:
        parts = re.split(r"\n+", seg)
        newline_split_segments.extend([p.strip() for p in parts if p.strip()])

    repaired = repair_segments(newline_split_segments)

    final_sentences = []

    for sent in repaired:
        sent = restore_abbreviations(sent)
        sent = normalize_sentence(sent)

        if is_bad_sentence(sent):
            continue

        final_sentences.append(sent)

    return final_sentences


# ============================================================
# Load MADLAD
# ============================================================

raw = load_dataset(
    "json",
    data_files={"train": URL},
    split="train",
)

print(raw)
print("Columns:", raw.column_names)
print("First row:", raw[0])


# ============================================================
# Detect text column
# ============================================================

possible_text_columns = ["text", "content", "sentence"]

text_col = None

for col in possible_text_columns:
    if col in raw.column_names:
        text_col = col
        break

if text_col is None:
    raise ValueError(f"Could not find text column. Available columns: {raw.column_names}")

print(f"Using text column: {text_col}")


# ============================================================
# Convert document rows to sentence rows
# ============================================================

rows = []

for doc_id, example in enumerate(raw):
    doc_text = example[text_col]
    sentences = split_document_to_sentences(doc_text)

    for sent_id, sentence in enumerate(sentences):
        rows.append({
            "id": f"madlad_bbc_{doc_id}_{sent_id}",
            "lang": LANG,
            "text": sentence,
            "source": SOURCE_NAME,
        })

df = pd.DataFrame(rows)

print("Before dedup:", len(df))

if len(df) > 0:
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("After dedup:", len(df))

if len(df) > 0:
    df["length"] = df["text"].str.len()
    df["words"] = df["text"].str.split().str.len()

    print(df[["text", "length", "words"]].head(20))
    print(df["length"].describe())
    print(df["words"].describe())

    df = df.drop(columns=["length", "words"])


# ============================================================
# Save
# ============================================================

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"Saved to: {OUTPUT_PATH}")

/Users/nicholas/miniconda3/envs/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 12941.03it/s]


Dataset({
    features: ['text'],
    num_rows: 1274
})
Columns: ['text']
First row: {'text': 'Jamita Evangelium, 15 September 2019, Minggu XIII Dung Trinitatis "Mangolu di Bagasan Serep ni Roha" (Lukas 14: 7-11) Pelita Batak Online\\nDi tongatonga ni parsaoran hajolmaon sian narobi sahat tu nuaeng on, sai dipodahon angka natuatua dohot pangajari do hinaringkot ni serep ni roha, asa tubu parsaoran nauli sama jolma i. Laos dipodahon do tong asa pasidingon ni jolma i na maralo tu serep ni roha i, i ma ginjang ni roha, ai sega do parsaoran nauli binahen ni ginjang ni roha. Nang di tongatonga ni parsaoran ni bangso ni Debata, Israel, songon na nalnal di buku habisuhon dipodahon do asa didungdung bangso i serep ni roha, jala dipasiding ginjang ni roha. Laho mangondolhon i, tangkas ma didok na dialo Debata do ginjang ni roha. "Ibana mamarengkelengkelihon halak pangelesi, alai dilehon do asi ni roha anggo tu halak na unduk marroha"(Poda 3: 34).\\nSongon i do nang di tongatonga ni parsaoran Ha

KeyboardInterrupt: 

In [3]:
from pathlib import Path
import re
import pandas as pd

from datasets import load_dataset
from sacremoses import MosesPunctNormalizer


# ============================================================
# Config
# ============================================================

URL = "https://huggingface.co/datasets/allenai/MADLAD-400/resolve/main/data/bbc/bbc_clean_0000.jsonl.gz"

OUTPUT_PATH = Path("sentence_pairs_data/madlad_bbc_sentences_regex_moses_clean.jsonl")

LANG = "bbc_Latn"
SOURCE_NAME = "madlad-400"

MIN_CHARS = 5
MAX_CHARS = 300
MIN_WORDS = 2

# Batak / Indonesian / common abbreviations that should not trigger sentence split
ABBREVIATIONS = {
    # Batak / church / title / name-related
    "Gr.", "Lbn.", "Pdt.", "Op.", "Br.", "St.", "Amang.", "Inang.",

    # Indonesian/common titles
    "Dr.", "Ir.", "Prof.", "H.", "Hj.", "Ny.", "Tn.", "Bpk.", "Bp.",
    "Sdr.", "Sdri.",

    # Places/admin
    "Jl.", "No.", "Ds.", "Kec.", "Kab.", "Prov.",

    # Common lowercase abbreviations
    "dll.", "dst.", "dsb.", "alm.",
}

DOT_PLACEHOLDER = "<DOT>"


# ============================================================
# Load Moses punctuation normalizer
# ============================================================

mpn = MosesPunctNormalizer(lang="en")


# ============================================================
# Quote-aware regex sentence pattern
# ============================================================

# This pattern captures:
# - normal sentence endings: .
# - question/exclamation endings: ? !
# - optional closing quote after punctuation: ."  .”  .'  .’
#
# It avoids splitting immediately after abbreviation periods because
# abbreviations are protected before this regex is applied.
#
# Example:
#   Didok ibana "Horas ma hamu." tu angka dongan.
#   -> one sentence
#
#   Didok ibana, "Horas ma hamu." Dung i laho ibana.
#   -> two sentences
SENTENCE_RE = re.compile(
    r"""
    .+?                         # sentence content, non-greedy
    [.!?]                       # sentence-ending punctuation
    (?:"|'|”|’)?                # optional closing quote
    (?=
        \s+[A-Z0-9"“']          # likely next sentence starts
        |
        \s*$                    # or end of line/text
    )
    |
    .+$                         # fallback final fragment without punctuation
    """,
    re.VERBOSE | re.DOTALL,
)


# ============================================================
# Text normalization
# ============================================================

def normalize_raw_text(text: str) -> str:
    """
    Normalize document-level raw text before sentence splitting.

    Important:
    - preserve newlines before splitting, because they can be useful boundaries
    - convert escaped \\n into real newlines
    """
    if text is None:
        return ""

    text = str(text)

    # Convert literal escaped newline into real newline
    text = text.replace("\\n", "\n")

    # Normalize newline variants
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Moses punctuation normalization
    text = mpn.normalize(text)

    # Normalize spaces/tabs but preserve newlines
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive newlines
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def normalize_sentence(text: str) -> str:
    """
    Normalize final sentence-level text.

    Important:
    - final training rows should not contain newlines
    - remaining line breaks are converted to spaces
    """
    if text is None:
        return ""

    text = str(text)

    # Final sentence should not preserve line breaks
    text = text.replace("\\n", " ")
    text = text.replace("\n", " ")

    # Moses punctuation normalization again after merging
    text = mpn.normalize(text)

    # Normalize all whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# Abbreviation protection
# ============================================================

def protect_abbreviations(text: str) -> str:
    """
    Protect abbreviation periods from sentence splitting.

    Example:
        Gr. Frederik Lbn. Gaol
    becomes:
        Gr<DOT> Frederik Lbn<DOT> Gaol
    """
    for abbr in sorted(ABBREVIATIONS, key=len, reverse=True):
        protected = abbr.replace(".", DOT_PLACEHOLDER)

        # Token-like match, not inside another word
        text = re.sub(
            rf"(?<!\w){re.escape(abbr)}",
            protected,
            text,
        )

    return text


def restore_abbreviations(text: str) -> str:
    return text.replace(DOT_PLACEHOLDER, ".")


def ends_with_known_abbreviation(text: str) -> bool:
    """
    Check whether a segment ends with a known abbreviation after restoration.
    """
    text = restore_abbreviations(text).strip()

    for abbr in ABBREVIATIONS:
        if text.endswith(abbr):
            return True

    return False


# ============================================================
# Name / fragment repair
# ============================================================

def looks_like_name_fragment(text: str) -> bool:
    """
    Detect short name-like fragments such as:
        Frederik Lbn.
        Gaol sian Buntu Raja

    This helps merge:
        Bendahara : Gr.
        Frederik Lbn.
        Gaol sian Buntu Raja

    into:
        Bendahara : Gr. Frederik Lbn. Gaol sian Buntu Raja
    """
    text = restore_abbreviations(text).strip()

    if not text:
        return False

    words = text.split()

    if len(words) > 6:
        return False

    allowed_lower = {
        "sian", "di", "ni", "tu", "bin", "boru", "br", "van", "de"
    }

    abbreviation_forms = set()
    for abbr in ABBREVIATIONS:
        abbreviation_forms.add(abbr)
        abbreviation_forms.add(abbr.rstrip("."))

    good = 0

    for word in words:
        clean = re.sub(r"^[^\w]+|[^\w.:'-]+$", "", word)

        if not clean:
            continue

        # Known abbreviation
        if clean in abbreviation_forms:
            good += 1
            continue

        # Capitalized token, likely name/place
        if re.match(r"^[A-Z][A-Za-z.'-]*\.?$", clean):
            good += 1
            continue

        # Lowercase connector inside names/phrases
        if clean.lower() in allowed_lower:
            good += 1
            continue

    return good >= max(1, len(words) - 1)


def should_merge(prev: str, curr: str) -> bool:
    """
    Decide whether two candidate sentence segments should be merged.
    """
    prev_restored = restore_abbreviations(prev).strip()
    curr_restored = restore_abbreviations(curr).strip()

    if not prev_restored or not curr_restored:
        return False

    # Previous segment ends with known abbreviation/title/name abbreviation
    if ends_with_known_abbreviation(prev_restored):
        return True

    # Example:
    # Bendahara : Gr.
    # Frederik Lbn.
    if re.search(r":\s*(Gr|Dr|Ir|Prof|Pdt|Op|St|Bpk|Bp)\.$", prev_restored):
        return True

    # Previous ends with colon and current is probably name/value continuation
    if prev_restored.endswith(":") and looks_like_name_fragment(curr_restored):
        return True

    # Previous contains role marker and current is name-like
    if ":" in prev_restored and looks_like_name_fragment(curr_restored):
        return True

    # Very short previous fragment followed by name-like fragment
    # Example:
    # Gr.
    # Frederik Lbn. Gaol
    if len(prev_restored.split()) <= 2 and looks_like_name_fragment(curr_restored):
        return True

    return False


def repair_segments(segments):
    """
    Merge over-split segments caused by line breaks or abbreviation/name fragments.
    """
    repaired = []

    for seg in segments:
        seg = seg.strip()

        if not seg:
            continue

        if not repaired:
            repaired.append(seg)
            continue

        if should_merge(repaired[-1], seg):
            repaired[-1] = repaired[-1].rstrip() + " " + seg.lstrip()
        else:
            repaired.append(seg)

    return repaired


# ============================================================
# Regex sentence splitting
# ============================================================

def regex_sentence_split(text: str):
    """
    Lightweight sentence splitter.

    Steps:
    1. Split by newlines first.
    2. Apply quote-aware sentence regex inside each line.
    3. Return candidate sentence segments.

    Abbreviations must already be protected before this function.
    """
    segments = []

    # Newlines are possible boundaries, but repair_segments can merge them later
    lines = re.split(r"\n+", text)

    for line in lines:
        line = line.strip()

        if not line:
            continue

        parts = SENTENCE_RE.findall(line)

        for part in parts:
            part = part.strip()
            if part:
                segments.append(part)

    return segments


# ============================================================
# Filtering
# ============================================================

def is_bad_sentence(sent: str) -> bool:
    """
    Basic sentence-level filtering for MT data.
    """
    if not sent:
        return True

    if len(sent) < MIN_CHARS:
        return True

    if len(sent) > MAX_CHARS:
        return True

    if len(sent.split()) < MIN_WORDS:
        return True

    # URLs, emails, HTML
    if re.search(r"http\S+|www\.\S+|\S+@\S+|<[^>]+>", sent, flags=re.IGNORECASE):
        return True

    # Mostly punctuation/symbol noise
    punct_count = sum(1 for c in sent if not c.isalnum() and not c.isspace())
    punct_ratio = punct_count / max(len(sent), 1)

    if punct_ratio > 0.35:
        return True

    # Very little alphabetic content
    alpha_count = sum(1 for c in sent if c.isalpha())
    alpha_ratio = alpha_count / max(len(sent), 1)

    if alpha_ratio < 0.45:
        return True

    return False


# ============================================================
# Main document splitter
# ============================================================

def split_document_to_sentences(text: str):
    """
    Full processing pipeline:

    raw document text
    -> normalize raw text
    -> protect abbreviations
    -> regex sentence split
    -> repair abbreviation/name fragments
    -> restore abbreviations
    -> normalize final sentence
    -> filter
    """
    text = normalize_raw_text(text)

    if not text:
        return []

    # Protect periods inside abbreviations before regex split
    text = protect_abbreviations(text)

    # Sentence split
    segments = regex_sentence_split(text)

    # Repair oversplitting across lines/name fragments
    segments = repair_segments(segments)

    final_sentences = []

    for sent in segments:
        sent = restore_abbreviations(sent)
        sent = normalize_sentence(sent)

        if is_bad_sentence(sent):
            continue

        final_sentences.append(sent)

    return final_sentences


# ============================================================
# Load MADLAD
# ============================================================

raw = load_dataset(
    "json",
    data_files={"train": URL},
    split="train",
)

print(raw)
print("Columns:", raw.column_names)
print("First row:", raw[0])


# ============================================================
# Detect text column
# ============================================================

possible_text_columns = ["text", "content", "sentence"]

text_col = None

for col in possible_text_columns:
    if col in raw.column_names:
        text_col = col
        break

if text_col is None:
    raise ValueError(f"Could not find text column. Available columns: {raw.column_names}")

print(f"Using text column: {text_col}")


# ============================================================
# Quick tests
# ============================================================

test_1 = (
    "Lam tuk ibana mangadopi angka tantangan.\\n"
    "Ndang muraura ibana muruk jala ndang madabu ibana tu kondisi stress."
)

test_2 = (
    "Bendahara : Gr.\n"
    "Frederik Lbn.\n"
    "Gaol sian Buntu Raja"
)

test_3 = 'Didok ibana "Horas ma hamu." tu angka dongan.'

test_4 = 'Didok ibana, "Horas ma hamu." Dung i laho ibana.'

print("\nTest 1:")
print(split_document_to_sentences(test_1))

print("\nTest 2:")
print(split_document_to_sentences(test_2))

print("\nTest 3:")
print(split_document_to_sentences(test_3))

print("\nTest 4:")
print(split_document_to_sentences(test_4))


# ============================================================
# Convert document rows to sentence rows
# ============================================================

rows = []

for doc_id, example in enumerate(raw):
    doc_text = example[text_col]
    sentences = split_document_to_sentences(doc_text)

    for sent_id, sentence in enumerate(sentences):
        rows.append({
            "id": f"madlad_bbc_{doc_id}_{sent_id}",
            "lang": LANG,
            "text": sentence,
            "source": SOURCE_NAME,
        })

    if (doc_id + 1) % 10000 == 0:
        print(f"Processed {doc_id + 1} documents, collected {len(rows)} sentences")


df = pd.DataFrame(rows)

print("\nBefore dedup:", len(df))

if len(df) > 0:
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("After dedup:", len(df))


# ============================================================
# Stats
# ============================================================

if len(df) > 0:
    df["length"] = df["text"].str.len()
    df["words"] = df["text"].str.split().str.len()

    print("\nSample output:")
    print(df[["text", "length", "words"]].head(20))

    print("\nLength stats:")
    print(df["length"].describe())

    print("\nWord stats:")
    print(df["words"].describe())

    df = df.drop(columns=["length", "words"])


# ============================================================
# Save
# ============================================================

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_json(
    OUTPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"\nSaved to: {OUTPUT_PATH}")

Dataset({
    features: ['text'],
    num_rows: 1274
})
Columns: ['text']
First row: {'text': 'Jamita Evangelium, 15 September 2019, Minggu XIII Dung Trinitatis "Mangolu di Bagasan Serep ni Roha" (Lukas 14: 7-11) Pelita Batak Online\\nDi tongatonga ni parsaoran hajolmaon sian narobi sahat tu nuaeng on, sai dipodahon angka natuatua dohot pangajari do hinaringkot ni serep ni roha, asa tubu parsaoran nauli sama jolma i. Laos dipodahon do tong asa pasidingon ni jolma i na maralo tu serep ni roha i, i ma ginjang ni roha, ai sega do parsaoran nauli binahen ni ginjang ni roha. Nang di tongatonga ni parsaoran ni bangso ni Debata, Israel, songon na nalnal di buku habisuhon dipodahon do asa didungdung bangso i serep ni roha, jala dipasiding ginjang ni roha. Laho mangondolhon i, tangkas ma didok na dialo Debata do ginjang ni roha. "Ibana mamarengkelengkelihon halak pangelesi, alai dilehon do asi ni roha anggo tu halak na unduk marroha"(Poda 3: 34).\\nSongon i do nang di tongatonga ni parsaoran Ha